In [2]:
# Generate high-res 10s images from DAS (single 10s files) with explicit distance masking
import os
import glob
import math
import h5py
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# ---- Parameters (10s windows, cutoff to 28 km max) ----
INPUT_DIR = "../data/GC_data"
OUTPUT_DIR = "unseen_data"
WINDOW_SECONDS = 10                    # target window length
FS_ASSUMED = 625.0                     # Hz (sampling frequency from metadata)
DX_ASSUMED_M = 1.0213001907746815      # meters per channel (from metadata)
DIST_MIN_KM = 4                        # lower cutoff
DIST_MAX_KM = 14                       # upper cutoff per request
VMAX = 2000.0                          # visualization dynamic range
IMG_H, IMG_W = 512, 1024               # high resolution output
CMAP = "jet"
DPI = 150                               # DPI
START_FROM_TIME = "000000"             # hardcode start filename HHMMSS (inclusive)

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Helper: parse HHMMSS from filename; fall back to header start time
def parse_time_from_filename(path):
    name = os.path.basename(path)
    ts = os.path.splitext(name)[0]
    try:
        return datetime.strptime(ts, "%H%M%S")
    except Exception:
        return None

# Find files (sorted by name HHMMSS.hdf5 assumed)
all_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.hdf5")))
if not all_files:
    raise FileNotFoundError(f"No HDF5 files found in {INPUT_DIR}")

# Filter to start from START_FROM_TIME
try:
    start_dt = datetime.strptime(START_FROM_TIME, "%H%M%S")
    file_paths = [fp for fp in all_files if (parse_time_from_filename(fp) or datetime.min) >= start_dt]
except Exception:
    # If parsing fails, fall back to all files
    file_paths = all_files

print(f"Processing {len(file_paths)} files starting from {START_FROM_TIME} (inclusive)")

saved = 0

for p in file_paths:
    base = os.path.splitext(os.path.basename(p))[0]
    try:
        with h5py.File(p, "r") as f:
            if "strainrate" in f:
                arr = f["strainrate"]
            elif "data" in f:
                arr = f["data"]
            else:
                arr = None
            if arr is None or arr.ndim != 2:
                print(f"Skipping {base}: bad array")
                continue
            # Read only needed slice after building mask to avoid huge allocations
            T, C = arr.shape
            approx_duration = T / FS_ASSUMED
            if approx_duration < 9.0:
                print(f"Skipping {base}: not enough samples for one 10s window")
                continue

            distances_full_km = (np.arange(C) * DX_ASSUMED_M) / 1000.0
            available_min = float(distances_full_km[0])
            available_max = float(distances_full_km[-1])

            dmin_eff = max(DIST_MIN_KM, available_min)
            dmax_eff = min(DIST_MAX_KM, available_max)
            if dmax_eff <= dmin_eff:
                print(f"Skipping {base}: effective window invalid ({DIST_MIN_KM}-{DIST_MAX_KM} km, avail {available_min:.2f}-{available_max:.2f} km)")
                continue

            # Compute index bounds explicitly, then slice dataset directly
            ch_lo = int(math.ceil((dmin_eff * 1000.0) / DX_ASSUMED_M))
            ch_hi = int(math.floor((dmax_eff * 1000.0) / DX_ASSUMED_M)) + 1
            ch_lo = max(0, min(ch_lo, C-1))
            ch_hi = max(ch_lo+1, min(ch_hi, C))
            if ch_hi - ch_lo < 2:
                print(f"Skipping {base}: too few channels after distance bounds")
                continue

            window = arr[:, ch_lo:ch_hi].astype(np.float32)

        # Standardize
        window = np.clip(window, 0.0, VMAX) / VMAX

        # Render
        fig = plt.figure(figsize=(IMG_W / DPI, IMG_H / DPI), dpi=DPI)
        ax = fig.add_axes([0, 0, 1, 1])
        ax.axis('off')
        ax.imshow(window, aspect='auto', cmap=CMAP, vmin=0.0, vmax=1.0, interpolation='nearest')
        out_name = f"{base}_T0s.png"
        out_path = os.path.join(OUTPUT_DIR, out_name)
        fig.savefig(out_path, dpi=DPI, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
        saved += 1

        print(f"Processed {base}: distance used {dmin_eff:.2f}-{dmax_eff:.2f} km, channels {ch_lo}-{ch_hi} → {window.shape[1]} cols")

    except Exception as e:
        print(f"Error processing {base}: {e}")

print(f"Saved {saved} 10s high-res images to '{OUTPUT_DIR}'.")
print("Next: Manually inspect 'train' images and move clear ship signatures into a 'test' folder for anomaly validation.")


Processing 115 files starting from 000000 (inclusive)
Processed 000006: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000006: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000016: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000016: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000026: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000026: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000036: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000036: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000046: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000046: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000056: distance used 4.00-14.00 km, channels 3917-13709 → 9792 cols
Processed 000056: distance used 4.00-14.00 km, channels 3917-13709 → 9792 col